In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss, numpy as np

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
cross   = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

In [ ]:
# doc_texts (list[str]) and doc_embs (np.ndarray shape (N, d))
#    If not, embed once and cache to disk:
#    doc_embs = encoder.encode(doc_texts, convert_to_numpy=True, batch_size=1024, normalize_embeddings=True)

d = encoder.get_sentence_embedding_dimension()
index = faiss.IndexHNSWFlat(d, 16)   # m=16 (HNSW graph degree)
index.hnsw.efConstruction = 100
index.hnsw.efSearch = 64
index.add(doc_embs.astype('float32'))

In [ ]:
def search(query: str, k_retrieve=100, k_rerank=50, batch_size=64):
    q_vec = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q_vec, k_retrieve)
    cands = [doc_texts[i] for i in I[0]]
    pairs = [(query, c) for c in cands[:k_rerank]]
    scores = cross.predict(pairs, batch_size=batch_size)  # GPU
    order = np.argsort(-scores)
    return [(cands[i], float(scores[i])) for i in order]

# Example:
# results = search("best way to debounce requests in Go")
# for t, s in results[:5]: print(round(s,3), t[:120])